# Pyomo starter template

The smallest complete Pyomo model: two variables, an objective, two constraints, solved with HiGHS. Copy it and change the numbers, variables and constraints for your own model.

Run the first cell, then the model cell. On your computer the same code is in `pyomo_starter_template.py`.

**The example model:** maximise $5x_0 + 7x_1$ subject to $3x_0 + 2x_1 \le 144$ and $x_0 \ge 4x_1$, with $x_0, x_1 \ge 0$.

**What you should see:** first the model printed back (check it is the model you meant), then the solution: $x_0 \approx 41.14$, $x_1 \approx 10.29$ and objective $\approx 277.71$.

In [ ]:
# Run this cell first.
# On the course website (JupyterLite) it installs this notebook's packages into the browser tab
# (about a minute the first time) and adapts them to the browser.
# On your own computer it does nothing: install everything once with
#     pip install -r requirements.txt      (the file is at the top of the Assignments folder)
# In Google Colab, run  !pip install pyomo highspy  instead.
import sys
if sys.platform == "emscripten":  # only in the browser
    import micropip
    await micropip.install(["pyomo==6.10.1", "highspy"])
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True
print("Ready.")

## The model

Each line of the next cell is one part of the mathematical model: the decision variables, the objective, then one `Constraint` per constraint. `model.pprint()` prints the model so you can check it, `solver.solve(...)` hands it to HiGHS, and `model.display()` prints the optimal values.

In [ ]:
from pyomo.environ import *

model = ConcreteModel()

model.x = Var(range(2), domain=NonNegativeReals)  # decision variables x[0], x[1] >= 0

model.obj = Objective(expr=5*model.x[0]+7*model.x[1], sense=maximize)  # maximise 5 x0 + 7 x1

model.constraint1 = Constraint(expr=3*model.x[0] + 2*model.x[1] <= 144)  # resource limit
model.constraint2 = Constraint(expr=model.x[0] >= 4*model.x[1])  # x0 at least four times x1

print("Displaying model #################\n")
model.pprint()

# solve the optimisation problem
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False) # set tee=True to display solver output

print()

print("Displaying solution #################\n")
model.display()

### Try it

Change the resource limit from 144 to 150 and run the model cell again. How does the solution change?

<details><summary>Answer</summary>

Both constraints stay binding, so the optimum moves along the line $x_0 = 4x_1$: $x_0 \approx 42.86$, $x_1 \approx 10.71$, objective $\approx 289.29$. Everything grows by the factor 150/144, so each extra unit of the resource is worth about 1.93 here.
</details>